# Day 20 — Python Package Manager + Week 4 Capstone

**How to use this notebook:**
- Read each exercise, fill in your solution (replace `pass`)
- Run the test cell right after to see **APPROVED** or **FAILED**
- The Capstone ties together everything from Days 16–20
- Do NOT modify the test cells

**Key concepts today:** `pip`, standard library packages: `os`, `csv`, `json`, `datetime`, `statistics`

---
## Exercise 1 — OS Module

Use the `os` module to write a function that inspects a directory.

Return a dict with:
- `'cwd'`: current working directory (string)
- `'files'`: sorted list of `.py` and `.ipynb` files in the current directory
- `'file_count'`: total number of those files

**Hint:** `os.getcwd()`, `os.listdir()`, check extensions with `str.endswith()`

In [1]:
import os

def inspect_directory():
    file = sorted([i for i in os.listdir() if i.endswith('.py') or i.endswith('.ipynb')])
    return{

        'cwd': os.getcwd(),
        'files': file,
        'file_count': len(file)

    }

In [2]:
# TEST CELL — do not modify
import os
def _check(label, got, expected):
    if got == expected:
        print(f'  ✔ APPROVED — {label}')
    else:
        print(f'  ✘ FAILED   — {label}')
        print(f'             got:      {got!r}')
        print(f'             expected: {expected!r}')

print('\n── Exercise 1: inspect_directory ──')
result = inspect_directory()
_check('cwd is string',     isinstance(result.get('cwd'), str),        True)
_check('files is list',     isinstance(result.get('files'), list),      True)
_check('files are sorted',  result.get('files') == sorted(result.get('files',[])), True)
_check('count matches',     result.get('file_count') == len(result.get('files',[])), True)


── Exercise 1: inspect_directory ──
  ✔ APPROVED — cwd is string
  ✔ APPROVED — files is list
  ✔ APPROVED — files are sorted
  ✔ APPROVED — count matches


---
## Exercise 2 - JSON Read/Write

Write a function that:
1. Takes a filename and a dictionary
2. Writes the dictionary to the file as JSON
3. Reads it back and returns the loaded dictionary
4. Deletes the file after reading

| Input | Expected Output |
|---|---|
| json_roundtrip('data.json', {'name': 'Alice', 'age': 25}) | {'name': 'Alice', 'age': 25} |

In [3]:
import json, os

def json_roundtrip(filename, data):
    with open(filename, 'w') as f: #write lines
        json.dump(data,f)
    with open(filename, 'r') as f: #read lines
        load = json.load(f)
        return load 
        
        
    os.remove(filename)
    
    
    

In [4]:
def _check(label, got, expected):
    if got == expected:
        print(f'  APPROVED - {label}')
    else:
        print(f'  FAILED   - {label}')
        print(f'             got:      {got!r}')
        print(f'             expected: {expected!r}')

print('-- Exercise 2: json_roundtrip --')
_check('roundtrip', json_roundtrip('data.json', {'name': 'Alice', 'age': 25}),
       {'name': 'Alice', 'age': 25})

-- Exercise 2: json_roundtrip --
  APPROVED - roundtrip


---
## ⭐ Week 4 Capstone — Weather Journal

This capstone ties together **datetime, exception handling, regex, file handling, and the os module** from Days 16–20.

You will build a weather journaling system that stores and analyzes weather data in a CSV file.

---

### `log_weather(filename, date_str, city, temp_c, condition)`
- Validate that `date_str` matches format `'YYYY-MM-DD'` (use regex or strptime)
- If invalid date → return `{'status': 'error', 'message': 'Invalid date format'}`
- Validate `temp_c` is a number between -90 and 60
- If invalid temp → return `{'status': 'error', 'message': 'Invalid temperature'}`
- Append valid entry to CSV with header: `date,city,temp_c,condition`
- Return `{'status': 'ok'}`

### `read_weather(filename)`
- Return all entries as list of dicts (temp_c as float)

### `weekly_summary(filename, city)`
- Filter entries for the given city (case-insensitive)
- Return `{'city': city, 'entries': n, 'avg_temp': float rounded to 1, 'hottest': float, 'coldest': float}`
- If no entries for that city → return `{'status': 'error', 'message': 'No data for city'}`

In [7]:
import csv
import os
import re
from datetime import datetime

def log_weather(filename, date_str, city, temp_c, condition):
    if re.match(r'\d{4}-\d{2}-\d{2}', date_str) is None:
        return {'status': 'error', 'message': 'Invalid date format'}
    if temp_c < -90 or temp_c > 60:
        return {'status': 'error', 'message': 'Invalid temperature'}
    
    if os.path.exists(filename):
        with open(filename, 'a') as f:
            writer = csv.DictWriter(f, fieldnames=['date', 'city', 'temp_c', 'condition'])
            writer.writerow({'date': date_str, 'city': city, 'temp_c': temp_c , 'condition': condition})
    else:
        with open(filename, 'w') as f:
            writer = csv.DictWriter(f, fieldnames=['date', 'city', 'temp_c', 'condition'])
            writer.writeheader()
            writer.writerow({'date': date_str, 'city': city, 'temp_c': temp_c , 'condition': condition})
    return {'status': 'ok'}

def read_weather(filename):
    with open(filename) as f:
        reader = csv.DictReader(f)
        result = []
        for i in reader:
            i['temp_c'] = float(i['temp_c'])
            result.append(i)

        return result

def weekly_summary(filename, city):
    with open(filename) as f:
        reader = csv.DictReader(f)
        filtered = [i for i in reader if i['city'].lower() == city.lower()]
        if filtered == []:
            return {'status': 'error', 'message': 'No data for city'}
        temps = [float(entry['temp_c']) for entry in filtered]

        return {
            
            'city': city,
            'entries': len(filtered),
            'avg_temp': round(sum(temps) / len(temps), 1),
            'hottest': max(temps),
            'coldest': min(temps)

        }



In [8]:
# TEST CELL — do not modify
import os
def _check(label, got, expected):
    if got == expected:
        print(f'  ✔ APPROVED — {label}')
    else:
        print(f'  ✘ FAILED   — {label}')
        print(f'             got:      {got!r}')
        print(f'             expected: {expected!r}')

f = 'test_weather.csv'
if os.path.exists(f): os.remove(f)

print('\n── Capstone: Weather Journal ──')

# Valid entries
_check('log Istanbul 22',  log_weather(f,'2024-01-01','Istanbul',22,'Sunny'),    {'status':'ok'})
_check('log Istanbul 18',  log_weather(f,'2024-01-02','Istanbul',18,'Cloudy'),   {'status':'ok'})
_check('log Istanbul 25',  log_weather(f,'2024-01-03','Istanbul',25,'Sunny'),    {'status':'ok'})
_check('log London 10',    log_weather(f,'2024-01-01','London',10,'Rainy'),      {'status':'ok'})

# Validation errors
_check('bad date',   log_weather(f,'01-13-2024','X',20,'Sun'), {'status':'error','message':'Invalid date format'})
_check('bad temp',   log_weather(f,'2024-01-01','X',200,'Sun'),{'status':'error','message':'Invalid temperature'})

# Read
entries = read_weather(f)
_check('entry count',      len(entries), 4)
_check('temp is float',    isinstance(entries[0]['temp_c'], float), True)

# Summary
summary = weekly_summary(f, 'istanbul')
_check('city name',   summary.get('city'),     'istanbul')
_check('entry count', summary.get('entries'),  3)
_check('avg temp',    summary.get('avg_temp'), 21.7)
_check('hottest',     summary.get('hottest'),  25.0)
_check('coldest',     summary.get('coldest'),  18.0)

_check('unknown city', weekly_summary(f,'Tokyo'), {'status':'error','message':'No data for city'})

if os.path.exists(f): os.remove(f)


── Capstone: Weather Journal ──
  ✔ APPROVED — log Istanbul 22
  ✔ APPROVED — log Istanbul 18
  ✔ APPROVED — log Istanbul 25
  ✔ APPROVED — log London 10
  ✔ APPROVED — bad date
  ✔ APPROVED — bad temp
  ✔ APPROVED — entry count
  ✔ APPROVED — temp is float
  ✔ APPROVED — city name
  ✔ APPROVED — entry count
  ✔ APPROVED — avg temp
  ✔ APPROVED — hottest
  ✔ APPROVED — coldest
  ✔ APPROVED — unknown city
